# Olist delivery reliability · exploratory analysis
Run from the repository root. This notebook checks the source grain, missingness, outcome definition, distributions, weekly pattern, and destination-state differences before the dashboard design.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
sys.path.insert(0, str(Path.cwd()))
from delivery_metrics import load_orders, analysis_orders, summary, weekly_rates, state_rates
df = load_orders('data/olist_delivery_orders.csv')
print('Shape:', df.shape)
print('Unique order IDs:', df.order_id.nunique())
print('Data types:')
print(df.dtypes.to_string())
print('Missing values:')
print(df.isna().sum().to_string())
print('Order statuses:')
print(df.order_status.value_counts().to_string())

In [ ]:
print('Raw purchase dates:', df.order_purchase_timestamp.min(), 'to', df.order_purchase_timestamp.max())
print('Raw promise dates:', df.promise_date.min(), 'to', df.promise_date.max())
print('Delivered with valid dates:', df.eligible.sum())
orders = analysis_orders(df)
print('Default analysis window:', summary(orders))
print('Late-day distribution:')
print(orders.loc[orders.is_late, 'late_days'].describe(percentiles=[.5, .9, .95, .99]).to_string())
print('Orders per promise month:')
print(df.loc[df.eligible].groupby(df.promise_date.dt.to_period('M')).size().to_string())

The edge months have tiny counts or are near the extract boundary, so the dashboard defaults to March 2017–August 2018. Date comparison is at calendar-day grain because the promised timestamp is a date at midnight.

In [ ]:
weekly = weekly_rates(orders)
states = state_rates(orders, minimum_orders=200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
axes[0].plot(weekly.week, weekly.late_rate * 100, color='#087f79', linewidth=2)
axes[0].set(title='Weekly late-delivery rate', ylabel='Late orders / eligible orders (%)')
axes[0].tick_params(axis='x', rotation=40)
top = states.head(10).iloc[::-1]
axes[1].barh(top.customer_state, top.late_orders, color='#ed6a5a')
axes[1].set(title='Late orders by destination state (top 10)', xlabel='Late orders')
fig.tight_layout()
plt.show()
print(states.head(10).to_string(index=False))

The dashboard retains only the weekly trend and state comparison. It uses rate plus sample count to read the trend and late-order count for region prioritization. The geography chart is descriptive, not causal. Payment and review variables would require extra joins and would not change the logistics manager's first decision.